# Testing and Comparing Montbrió-Pazó-Roxin Model Differential Equations Implementations in TVB and VBjax


The differential equations (usually called `dfun` - derivative function - in the implementations) describing neural‑mass models involve many state variables and parameters. As such, they are prone to sign errors and other algebraic slips. Because the variables in the code usually follow standard mathematical notation (viz. TVB, VBjax, Neurolib), a single mistyped character is an easy mistake to make. Their many parameters, and multiple terms make it difficult to spot mistakes by eye, and small errors could slip by a regular code review. For this reason, a simple way to write unit tests is useful and could save model developers valuable time.

This notebook/test suite focuses exclusively on unit-testing the `dfun` of the Montbrió-Pazó-Roxin (MPR) model, and provides two ways to test `dfun` correctness depending on the models implementation. It also provides several ways to pick test granularity, based on the users working knowledge of the model.

`dfun` is an important component of a model, and developers often expose it as a separate function. Both TVB and VBjax libraries utilize separate `dfun`. If this is the case, this tutorial provides a guideline on how build a test suite for a separate `dfun`. However, this is not a requirement for successful testing of `dfun`, and in this guide you will find a way to test it even if it is not a separate function.

In the case where `dfun` is not a separate function, the requirements are the same as for the all other tutorials in this suite. A chosen ground truth model to compare against, for the model to be able to be initialized with any relevant initial conditions and parameters, and for it to be able to do one step of Euler integration.

The first step is identifying the correspondce between the model parameters across the two simulators. For TVB and VBjax, most parameters map directly between the two implementations. However, the `Gamma` parameter in TVB is unused and can be disregarded.

For hardcoded parameters, set the values of the other simulator to those hardcoded values. If both simulators have the same parameter hardcoded to a different value, consider them inherently incompatible

In this case, `Gamma` parameter of the TVB implementation is irrelevant, and the default parameter values match between TVB and VBjax (`mpr_default_theta`), so these are used as the starting point. Take these parameters and make them explicit in the `Config` class, for reusability and version stability (as default values can change between versions).

In [1]:
from vbjax import mpr_default_theta

mpr_default_theta

c[vbjax] ███▒▒▒▒▒▒▒ loading
c[vbjax] (ﾉ☉ヮ⚆)ﾉ ⌒*:･ﾟ✧ can haz 16 cores
c[vbjax] ò_ô shtns is not available
c[vbjax] ᕕ(ᐛ)ᕗ ready


MPRTheta(tau=1.0, I=0.0, Delta=1.0, J=15.0, eta=-5.0, cr=1.0, cv=0.0)

## Choosing Test Values for `r` and `V`

Rather than manually selecting points on the phase plane—which is both time-consuming and error-prone, we chose a sufficient number of random points.

These are stored in `test_data/random_points0.0_2.0_-2.0_1.5.npz`, with `r` values in the range [0, 2] and `V` in [-2, 1.5]. The relevant parts of the phase plane can move away from these values of state variables, so for this purpose we've implemented a sanity check functionality.

The points are generated once per `(r, V)` range and stored for reproducibility. This is necessary because NumPy’s RNG does not guarantee cross-version stability—even with seeding.

Below is a visualisation of sampled points on 2 differently selected phase planes across 2 parameter setups.
The leftmost one is the default parameters and selected range.

![r_v comparison on phase planes](Pictures/svg/phase_planes.svg)

## Defining Test Cases

Next, we define our parameter sets. There are two approaches:
1. **Target key regions of the bifurcation diagram**
2. **Use sampling across the parameter space with an appropriate heuristic**

The recommended approach is to use both, as option #1 is much more likely to find issues as it puts the model in modes that are similar to real operation.

ption #1 as it is most likely to find issues with the functions, as it expects predictable, yet different behavior of the function. However, the regions in parameter space need to be hand selected based on prior experimentation or bifurcation analysis. The problem that arises from this approach is that the testing has a manual component, which is undesirable.

Option #2 avoids the issue of manual checking by selecting, in this case, a default set of parameters that don't mask interactions between them. Then, for each parameter, a full range sweep is done while keeping the other parameters at the chosen default values. This approach should reveal issues with parameters that are sufficiently sensitive, but may miss issues with less sensitive parameters. There is also the issue of granularity, as if this is too low it may miss a problematic range of values entirely. However, as it is possible to fully automate this approach, it is worthwhile to implement nevertheless.



<!-- TODO REWRITE/FIX As for the one parameter at a time part of option two, this comes from the pragmatic view of the problem as simply a parametrized f(x). If there is a difference in f(x) and f'(x), changing multiple parameters at the same time (and with sufficient sampling density) should reveal this difference just the same as changing only one. -->

In [2]:
import numpy as np

# Option 1

_test_cases = [{"eta": [-10, -5, -0.001]}]
_test_cases += [{"tau": [0.25, 0.26]}]


# Option 2

_test_cases += [
    {"tau": np.linspace(0.001, 15.0, 100).tolist()},
    {"I": np.linspace(-10.0, 10.0, 100).tolist()},
    {"Delta": np.linspace(0.0, 10.0, 100).tolist()},
    {"J": np.linspace(-25.0, 25.0, 100).tolist()},
    {"eta": np.linspace(-10.0, 10.0, 100).tolist()},
    {"cr": np.linspace(0.0, 1.0, 100).tolist()},
    {"cv": np.linspace(0.0, 1.0, 100).tolist()},
]

## Parameter Expansion

Each test case is represented as a row in a Pandas DataFrame.

- The first few columns contain model parameters.
- `_low` and `_high` define the `(r, V)` sampling range for the test case.
- `linspace_size` specifies how many points to generate in that range.

This structure allows for different sampling regions depending on the parameter set.


In [3]:
from utils.mpr_tvb_vbjax_test_parameters import _expand_test_cases

test_cases = _expand_test_cases(_test_cases)
test_cases

,tau,I,Delta,J,eta,cr,cv,r_low,r_high,V_low,V_high,linspace_size
0,1.00,0.0,1.0,15.0,-10.000,1.0,0.000000,0.0,2.0,-2.0,1.5,100.0
1,1.00,0.0,1.0,15.0,-5.000,1.0,0.000000,0.0,2.0,-2.0,1.5,100.0
2,1.00,0.0,1.0,15.0,-0.001,1.0,0.000000,0.0,2.0,-2.0,1.5,100.0
3,0.25,0.0,1.0,15.0,-5.000,1.0,0.000000,0.0,2.0,-2.0,1.5,100.0
4,0.26,0.0,1.0,15.0,-5.000,1.0,0.000000,0.0,2.0,-2.0,1.5,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...
700,1.00,0.0,1.0,15.0,-5.000,1.0,0.959596,0.0,2.0,-2.0,1.5,100.0
701,1.00,0.0,1.0,15.0,-5.000,1.0,0.969697,0.0,2.0,-2.0,1.5,100.0
702,1.00,0.0,1.0,15.0,-5.000,1.0,0.979798,0.0,2.0,-2.0,1.5,100.0
703,1.00,0.0,1.0,15.0,-5.000,1.0,0.989899,0.0,2.0,-2.0,1.5,100.0


## Compatibility Layer

To be able to run the simulators with the given parameter combinations, we build a translation layer to map our test case data into the input formats expected by TVB and VBjax implementations.

In [ ]:
from typing import List
from numpy import typing as npt
import numpy as np
import pandas as pd
from tvb.simulator.models.infinite_theta import MontbrioPazoRoxin
from utils.mpr_tvb_vbjax_test_functions import load_or_generate_data_for_testcase
from utils.mpr_tvb_vbjax_test_parameters import coupling_
from vbjax import MPRTheta, mpr_dfun

parameter_list = ["tau", "I", "Delta", "J", "eta", "cr", "cv"]


def run_tvb_implementation(test_case, test_data: List[npt.NDArray]):

    # Extract parameters for the model from the test case
    parameters_only = test_case[parameter_list]

    # Transform parameters to a data format acceptable for the model
    param_dict = {k: np.array([v]) for k, v in parameters_only.items()}

    # Run the dfun
    return MontbrioPazoRoxin(**param_dict, Gamma=np.r_[0.0]).dfun(
        test_data,
        coupling=coupling_,
    )


def run_vbjax_implementation(test_case, test_data: List[npt.NDArray]):

    # Extract parameters for the model from the test case
    parameters_only = test_case[parameter_list]

    # Transform parameters to a data format acceptable for the model
    params_as_mprtheta = MPRTheta(**parameters_only.to_dict())

    # Run the dfun
    return mpr_dfun(
        test_data,
        c=coupling_,
        p=params_as_mprtheta,
    )


# Convenience wrapper for running both dfuncs on the same dataset with the same parameters
def run_test(test_case: pd.Series):

    test_data = load_or_generate_data_for_testcase(test_case)

    return (
        run_tvb_implementation(test_case, test_data),
        run_vbjax_implementation(test_case, test_data),
        # Here you could place more implementations in case you're testing multiple simulators
    )

ModuleNotFoundError: No module named 'model_testing'

## Testing without explicit `dfun`

If your model does not implement a separate `dfun`, you may test the simulator as a greybox. The first step is identifying the relevant `dfun` parameters, and setting them to the same values. MPR equations have many parameters, we'll show each in the example below. There are other model parameters relevant for simulation equivalence testing, such as Dt or initial values. For the reason of reproducibility we recommend writing a `Config` class, and a wrapper for each model, so that inputs and outputs match for each model. These can be viewed in full in files `config.py`, `tvb_mpr_model.py` and `vbjax_model.py`.

For parameters relevant to the computation that only appear in one model but not the other, choose values that allow consistent mapping between the models. (Good candidates include powers of ten for scaling, zero for additive terms, or identical "magic constants".)

In [ ]:
import numpy as np
import tvb.simulator.lab as tvbl


class Config:
    def __init__(self, initial_conditions_seed, noise_seed=42):
        self.dt = 0.01
        self.speed = 2.0
        self.init_cond_rng = np.random.default_rng(seed=initial_conditions_seed)
        self.history_length = 10
        self.coupling_strength = 1.0
        self.noise = 0.0
        self.noise_seed = noise_seed
        self.conn = None

    def init_cond_for_dfun(self):
        self.conn = tvbl.connectivity.Connectivity()
        self.size = 1000
        self.conn.weights = np.zeros((self.size, self.size))
        self.conn.tract_lengths = np.zeros((self.size, self.size))
        self.conn.centres_spherical(number_of_regions=self.size)
        self.conn.compute_region_labels()
        self.conn.try_compute_hemispheres()
        self.conn.configure()
        self.init_cond = np.r_[[[np.ones((self.size, 1)), np.ones((self.size, 1))]]]

We'll extend the generic config class to make it usable for MPR model specifically

In [ ]:
class MPRConfig(Config):
    def __init__(self, initial_conditions_seed=42, noise_seed=42):
        super().__init__(initial_conditions_seed=initial_conditions_seed, noise_seed=noise_seed)
        self.tau = 1.0
        self.I = 0.0
        self.Delta = 1.0
        self.J = 15.0
        self.eta = -5.0
        self.cr = 1.0
        self.cv = 0.0

    def init_cond_for_dfun(self):
        super().init_cond_for_dfun()
        self.init_cond = np.r_[
            [
                [
                    self.init_cond_rng.uniform(0.0, 2.0, (self.size, 1)),
                    self.init_cond_rng.uniform(-2.0, 1.5, (self.size, 1)),
                ]
            ]
        ]

### TVB wrapper

This wrapper maps the config class properties to the TVB simulator initialization. It also removes run metadata from the simulator result.

In [ ]:
from tvb.simulator import simulator, coupling
from tvb.simulator.integrators import EulerStochastic
from tvb.simulator.monitors import Raw
from tvb.simulator.models.infinite_theta import MontbrioPazoRoxin
import tvb.simulator.lab as tvbl


class TvbMPRModel:
    def __init__(self, config: MPRConfig):
        self.config = config
        self._configure_sim()

    def _configure_sim(self):
        self.sim = simulator.Simulator(
            connectivity=self.config.conn,
            model=MontbrioPazoRoxin(
                tau=np.r_[self.config.tau],
                I=np.r_[self.config.I],
                Delta=np.r_[self.config.Delta],
                J=np.r_[self.config.J],
                eta=np.r_[self.config.eta],
                cr=np.r_[self.config.cr],
                cv=np.r_[self.config.cv],
            ),
            integrator=EulerStochastic(
                dt=self.config.dt,
                noise=tvbl.noise.Additive(
                    nsig=np.r_[self.config.noise],
                    noise_seed=self.config.noise_seed,
                ),
            ),
            initial_conditions=self.config.init_cond,
            conduction_speed=self.config.speed,
            monitors=[Raw()],
            simulation_length=self.config.dt,
            coupling=coupling.Scaling(a=np.r_[self.config.coupling_strength]),
        )
        self.sim.configure()

    def run(self):
        return self.sim.run()[0][1]

### VBJax wrapper

This wrapper is more involved. VBJax doesn't have a `Simulator` equivalent, instead the user builds the handling of connectivity for themselves. Example below follows other VBJax models in this regard.

In [ ]:
import jax.numpy as jp
import vbjax as vb


class VBJaxModel:
    def __init__(self, config: MPRConfig):
        self.config = config
        self.parameters = vb.MPRTheta(
            tau=config.tau,
            I=config.I,
            Delta=config.Delta,
            J=config.J,
            eta=config.eta,
            cr=config.cr,
            cv=config.cv,
        )

    def run(self):

        # In VBJax, connectivity handling is up to the user
        # Here's an implementation mirroring other VBJax models
        def network(state, _):
            coupling = self.config.coupling_strength * self.config.conn.weights @ state.T
            coupling = coupling.T
            return vb.mpr_dfun(state, coupling, self.parameters)

        squeezed_init_conds = jp.squeeze(self.config.init_cond)

        noise_array = vb.randn(*squeezed_init_conds.shape)

        step, _ = vb.make_sde(
            dt=self.config.dt,
            dfun=network,
            gfun=self.config.noise,
            return_euler=True,
        )

        # step returns tuple [euler result, heun result], extract euler
        return step(squeezed_init_conds, noise_array, self.parameters)[0]

### Test definition

As mentioned prior, we utilize two approaches. The first one where uses the bifurcation diagram to test known modes given set parameters. The second does a parameter sweep around the known point, a uniform random sampling of 10% of the recommended parameter range.

In [ ]:
tau_range = {"lo": 0.001, "hi": 15.0, "step": 0.01}
I_range = {"lo": -10.0, "hi": 10.0, "step": 0.01}
Delta_range = {"lo": 0.0, "hi": 10.0, "step": 0.01}
J_range = {"lo": -25.0, "hi": 25.0, "step": 0.0001}
eta_range = {"lo": -10.0, "hi": 10.0, "step": 0.0001}
cr_range = {"lo": 0.0, "hi": 1, "step": 0.1}
cv_range = {"lo": 0.0, "hi": 1, "step": 0.1}


def range_around(param_val, param_range, percentage):
    total_range = abs(param_range["hi"] - param_range["lo"])
    five_percent_range = total_range * percentage * 0.01
    _range = np.arange(
        param_val - five_percent_range,
        param_val + five_percent_range,
        0.01,
        dtype=float,
    )
    _range = _range[_range >= param_range["lo"]]
    _range = _range[_range <= param_range["hi"]]
    _range = np.round(_range, 2)
    return _range

In [ ]:
def build_test_case(config: MPRConfig):
    param_vals = [config.tau, config.I, config.Delta, config.J, config.eta, config.cr, config.cv]
    param_ranges = [tau_range, I_range, Delta_range, J_range, eta_range, cr_range, cv_range]
    return [range_around(val, r, percentage=5) for val, r in zip(param_vals, param_ranges)]

In [ ]:

def run_one_test(config):
    tvb = np.squeeze(TvbMPRModel(config).run())
    vbjax = VBJaxModel(config).run()
    res = round(tvb - vbjax, 6)
    try:
        np.testing.assert_allclose(tvb, vbjax, rtol=1e-1)
    except AssertionError as e:
        print(config.tau, config.I, config.Delta, config.J, config.eta, config.cr, config.cv)
        print(res[res != 0])
        print(e)

def sweep_param(config, param_name, test_range):
    # save original value
    default_value = getattr(config, param_name)

    for param_value in test_range: # np.arange(...)
        p_v_rounded = round(param_value, 2)
        setattr(config, param_name, p_v_rounded)
        config.init_cond_for_dfun()
        run_one_test(config)

    # restore default
    setattr(config, param_name, default_value)


def run_test(config: MPRConfig):
    run_one_test(config)
    tc = build_test_case(config)
    for _range, param_name in zip(tc, ["tau", "I", "Delta", "J", "eta", "cr", "cv"]):
        sweep_param(config, param_name, _range)

In [ ]:
config = MPRConfig()
config.coupling_strength = 0.0
config.init_cond_for_dfun()
run_test(config)

## On the Pitfalls of Shared Sampling Ranges

As shown below, using identical `(r, V)` ranges across all test cases may result in test outcomes from `dfun` regions where nothing of interest happens (e.g., false positives or negatives). It’s important to tailor the sampling region to the expected dynamics. As of now, we're unable to do this automatically, which is partly why we chose to go with a lower number of parameter combinations.


In [ ]:
current_test_case = default_values.iloc[0].copy()

current_test_case["V_low"] = 1.0
current_test_case["V_high"] = 3.0

print(current_test_case)

As you can see on the middle and right-most diagrams, badly selected combinations of parameter and state variable ranges may result in missing the stable points of the phase plane

![r_v comparison on phase planes](Pictures/svg/phase_planes.svg)

## Pytest: Minimal and Extended Testing

The following cells contain two `pytest`-based test setups:

1. A **minimal** version.
2. A **extended** version that logs test results when running all tests, and logs more data when specific tests are re-run.

These are provided for educational use. If you wish to modify them, make changes directly in the test scripts.


In [ ]:
# Minimal version

import numpy as np
import pytest

from model_testing.tutorials.utils.mpr_tvb_vbjax_test_functions import run_test
from model_testing.tutorials.utils.mpr_tvb_vbjax_test_parameters import test_parameters

indices, rows = zip(*test_parameters.iterrows())


@pytest.mark.parametrize("row", rows, ids=indices)
def test_one_case(row):
    test_results = run_test(row)
    assert np.allclose(*test_results)

In [ ]:
# Extended version

import json
import os

import numpy as np
import pytest
from model_testing.tutorials.utils.mpr_tvb_vbjax_test_functions import run_test

from model_testing.tutorials.utils.mpr_tvb_vbjax_test_parameters import test_parameters

indices, rows = zip(*test_parameters.iterrows())
results = []


@pytest.fixture(scope="session")
def is_single_test(pytestconfig):
    # All command line args after 'pytest'
    args = pytestconfig.args  # list of CLI args, e.g. ['test.py::test_one_case[6]']
    # If exactly one argument that contains a nodeid with param, consider it a single test run
    single_test = len(args) == 1 and ("::" in args[0])
    return single_test


@pytest.mark.parametrize("row", rows, ids=indices)
def test_one_case(row, is_single_test, request):
    test_results = run_test(row)
    try:
        assert np.allclose(*test_results)
        results.append("1")
    except AssertionError as identifier:
        results.append("0")
        if not is_single_test:
            raise identifier
        call_id = request.node.callspec.id
        with open(f"results/failed_test_id_{call_id}.json", "w", encoding="utf-8") as file:
            json.dump(
                {
                    "test_case": row.to_dict(),
                    "test_results": {
                        # Rename these as necessary
                        "tvb": test_results[0].tolist(),
                        "vbjax": test_results[1].tolist(),
                        # You can also add more if you alter run_test
                    },
                },
                file,
            )
        raise identifier


@pytest.fixture(scope="session", autouse=True)
def write_results(is_single_test):
    yield 0
    if is_single_test:
        return
    with open(f"results/all_test_results.csv", "w", encoding="utf-8") as file:
        file.write("test_results" + os.linesep)
        print(results)
        file.writelines(os.linesep.join(results))

## Running the Test Suite

Here is an example of how you could run the `pytest` suite directly from a notebook.

Note: This executes tests as defined in the scripts and **will not reflect** changes made interactively in the notebook.
You can find these scripts in the example directory.


In [ ]:
import pytest

# pytest.main(["mpr_tvb_vbjax_test.py"])